In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# read results file
base_path = Path().cwd()

filename = "conv_gpt-4o-mini_n-1_acc-0.833_10.24.2024-13:58:14"
filename += ".json"

file_under_investigation = base_path.parent / "results" / filename
with file_under_investigation.open("r") as f:
    data = json.load(f)

In [3]:
# split responses and general stats
responses = pd.DataFrame(data["responses"])
del data["responses"]
stats = pd.DataFrame({k: [v]for k,v in data.items()})

In [4]:
# fetch incorrect instances
responses = responses.loc[responses["accuracy"] == False].copy()
responses.drop(columns=["ai_response", "accuracy", 'completion_tokens', 'prompt_tokens', 'total_tokens'], inplace=True)
responses.set_index("idx", inplace=True)

In [5]:
# get original sensors index
data_path = base_path.parent / "data"
sensor_list = pd.read_excel(data_path / "requirements.xlsx").columns[1:]

In [6]:
# get label per vector
func = lambda x: " & ".join(sensor_list[np.array(x[1:-1].split(",")) == "1"].tolist())
responses["true_label"] = responses["true_vector"].apply(func)
responses["pred_label"] = responses["pred_vector"].apply(func)

In [7]:
# get new version number
versions = [int(f.name.split("-")[-1].replace(".xlsx", "")) for f in data_path.glob(f"incorrect-instances_{filename.split('_')[1]}-*.xlsx")]
versions.append(0)
new_version = max(versions) + 1

# save incorrect instances with labels
responses.to_excel(data_path / f"incorrect-instances_{filename.split('_')[1]}-{new_version}.xlsx")